In [ ]:
from pathlib import Path
import sys

repo_root = Path.cwd().resolve()
while repo_root != repo_root.parent and not (repo_root / "pyproject.toml").exists():
    repo_root = repo_root.parent

src_path = repo_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))


In [ ]:
from erddapy import servers
glider_servers = next(server for label, server in servers.items() if "gliders" in server.url)
glider_servers

Server(description='NOAA IOOS NGDAC (National Glider Data Assembly Center)', url='https://gliders.ioos.us/erddap/')

In [ ]:
from erddapy import servers
servers

{'voto': Server(description='Voice of the Ocean', url='https://erddap.observations.voiceoftheocean.org/erddap/'),
 'slgo-ogsl': Server(description='St. Lawrence Global Observatory - CIOOS | Observatoire global du Saint-Laurent - SIOOC', url='https://erddap.ogsl.ca/erddap/'),
 'cswc': Server(description='CoastWatch West Coast Node', url='https://coastwatch.pfeg.noaa.gov/erddap/'),
 'apdrc': Server(description='ERDDAP at the Asia-Pacific Data-Research Center', url='https://apdrc.soest.hawaii.edu/erddap/'),
 'ncei': Server(description="NOAA's National Centers for Environmental Information (NCEI)", url='https://www.ncei.noaa.gov/erddap/'),
 'bcodmo': Server(description='Biological and Chemical Oceanography Data Management Office (BCO-DMO) ERDDAP', url='https://erddap.bco-dmo.org/erddap/'),
 'emodnet': Server(description='European Marine Observation and Data Network (EMODnet) ERDDAP', url='https://erddap.emodnet.eu/erddap/'),
 'emodnet physics': Server(description='European Marine Observati

In [ ]:
from urllib.error import HTTPError

from erddap_client.erddap_wrapper import GliderIngestor

glider_ingestor = GliderIngestor()

grid_datasets = {}
no_dataset_grids = {}
grid_step_size = 10
for lat in range(0, 60, grid_step_size):
    for lon in range(-180, 180, grid_step_size):
        min_lat = lat
        max_lat = lat + grid_step_size
        min_lon = lon
        max_lon = lon + grid_step_size

        grid_label = f"lat_{min_lat}_{max_lat}_lon_{min_lon}_{max_lon}"

        try:
            grid_datasets[grid_label] = glider_ingestor.dataset_search(
                min_lat=min_lat,
                max_lat=max_lat,
                min_lon=min_lon,
                max_lon=max_lon,
            ).assign(grid_label=grid_label)
        except HTTPError:
            no_dataset_grids[grid_label] = {
                "min_lat": min_lat,
                "max_lat": max_lat,
                "min_lon": min_lon,
                "max_lon": max_lon,
            }

KeyboardInterrupt: 

In [ ]:
import pandas as pd

glider_datasets = pd.concat(grid_datasets.values(), ignore_index=True).drop_duplicates()
glider_datasets.columns

In [ ]:
def combine_grid_labels(labels):
    unique_labels = pd.Series(labels).dropna().drop_duplicates()
    return ", ".join(unique_labels)

metadata_columns = [column for column in glider_datasets.columns if column != "grid_label"]
glider_datasets = (
    glider_datasets.groupby("Dataset ID", as_index=False, sort=False)
    .agg({
        **{column: "first" for column in metadata_columns if column != "Dataset ID"},
        "grid_label": combine_grid_labels,
    })
)

glider_datasets = glider_datasets[['Dataset ID','Title','Summary','Institution','grid_label']]
glider_datasets.head()

In [ ]:
datasets_metadata_list = []

for dataset_title in glider_datasets['Title']:
    df_info = glider_ingestor.get_dataset_metadata(dataset_title)

    df_info['Title'] = dataset_title

    datasets_metadata_list.append(df_info)

datasets_metadata = pd.concat(datasets_metadata_list, ignore_index=True)
datasets_metadata[
    (datasets_metadata['Row Type'] == 'variable')
    & (
        (datasets_metadata['Variable Name'].str.contains('precise_lat'))
        | (datasets_metadata['Variable Name'].str.contains('precise_lon'))
        | (datasets_metadata['Variable Name'] == 'salinity')
        | (datasets_metadata['Variable Name'] == 'temperature')
    )
].head(8)

AttributeError: 'GliderIngestor' object has no attribute 'get_dataset_metadata'

In [ ]:
glider_dataset_count = len(datasets_metadata['Title'].unique())

lat_col = 'precise_lat'
lon_col = 'precise_lon'

row_count_for_coords = datasets_metadata[
    (datasets_metadata['Row Type'] == 'variable')
    & 
    (
        datasets_metadata['Variable Name'].str.contains(lat_col) 
        | datasets_metadata['Variable Name'].str.contains(lon_col) 
    ) 
].shape[0]

coord_count_per_row = 2

assert glider_dataset_count, (row_count_for_coords / coord_count_per_row)

In [ ]:
subset_grid_labels = [key for key in grid_datasets if '20_30' in key and '-90_-80' in key]
grid_of_interest = subset_grid_labels[0]

In [ ]:
last_index = -1

dataset_title = grid_datasets[grid_of_interest].iloc[last_index]['Dataset ID']
dataset_title

In [ ]:
glider_data = glider_ingestor.get_dataset(dataset_title)

glider_units = glider_data.iloc[:1, :]
glider_data_df = glider_data.iloc[1:, :]

In [ ]:
from erddap_client.mapping import make_map_axes
import cartopy.crs as ccrs
import matplotlib.pyplot as plt

fig, ax, track = make_map_axes(glider_data_df, lon_pad=10, lat_pad=3)

ax.plot(track[lon_col], track[lat_col], color="tab:red", linewidth=2, transform=ccrs.PlateCarree())
ax.scatter(track[lon_col].iloc[0], track[lat_col].iloc[0], color="green", s=60, label="Start", transform=ccrs.PlateCarree())
ax.scatter(track[lon_col].iloc[-1], track[lat_col].iloc[-1], color="black", s=60, label="End", transform=ccrs.PlateCarree())

ax.set_title(f"Glider Track: {dataset_title}")
ax.legend()
plt.show()

In [ ]:
salinity_col = 'salinity'

track_salinity = glider_data_df[[lon_col, lat_col, salinity_col]].copy()

for col in [lon_col, lat_col, salinity_col]:
    track_salinity[col] = pd.to_numeric(track_salinity[col], errors='coerce')

track_salinity = track_salinity.dropna()

fig, ax, _ = make_map_axes(track_salinity, lon_pad=10, lat_pad=10)

scatter = ax.scatter(
    track_salinity[lon_col],
    track_salinity[lat_col],
    c=track_salinity[salinity_col],
    cmap="viridis",
    s=18,
    transform=ccrs.PlateCarree(),
)
ax.plot(track_salinity[lon_col], track_salinity[lat_col], color="0.3", linewidth=0.8, alpha=0.5, transform=ccrs.PlateCarree())
ax.scatter(track_salinity[lon_col].iloc[0], track_salinity[lat_col].iloc[0], color="white", edgecolor="black", s=70, label="Start", transform=ccrs.PlateCarree())
ax.scatter(track_salinity[lon_col].iloc[-1], track_salinity[lat_col].iloc[-1], color="black", s=70, label="End", transform=ccrs.PlateCarree())

colorbar = plt.colorbar(scatter, ax=ax, pad=0.02, shrink=0.75)
colorbar.set_label(salinity_col)
ax.set_title(f"Glider Track Colored by Salinity: {dataset_title}")
ax.legend()
plt.show()